In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("Dados/dataset_limpo.csv")
pd.set_option('display.max_columns', None)

FileNotFoundError: [Errno 2] No such file or directory: 'Dados/dataset_limpo.csv'

In [ ]:
df.head()

,Id,Municipio,Sigla_Estado,Nota_CN,Nota_CH,Nota_LC,Nota_MT,Lingua_Estrangeira,Nota_Comp1,Nota_Comp2,Nota_Comp3,Nota_Comp4,Nota_Comp5,Nota_Redacao,Provas_CH/LC,Provas_CN/MT,Presenca_Dia_1,Presenca_Dia_2,Media_Objetivas,Media_Final,Regiao,Nivel
0,206403,Aratuba,CE,436.8,377.8,423.4,427.1,Espanhol,80.0,60.0,60.0,80.0,20.0,300.0,Azul,Amarela,1,1,416.275,393.02,Nordeste,Muito Baixo
1,3604651,Tijucas,SC,521.9,601.9,605.5,689.2,Ingles,160.0,200.0,200.0,180.0,180.0,920.0,Amarela,Cinza,1,1,604.625,667.70,Sul,Alto
2,1461268,Rolândia,PR,363.0,548.4,557.2,456.4,Espanhol,120.0,120.0,40.0,120.0,80.0,480.0,Branca,Azul,1,1,481.250,481.00,Sul,Baixo
3,4301058,Osasco,SP,550.7,553.8,605.9,629.1,Ingles,140.0,200.0,160.0,160.0,80.0,740.0,Verde,Verde,1,1,584.875,615.90,Sudeste,Médio
4,140768,Natal,RN,482.7,531.6,439.2,496.7,Espanhol,100.0,80.0,60.0,120.0,0.0,360.0,Branca,Azul,1,1,487.550,462.04,Nordeste,Baixo


Calculo da probabilidade a priori

# Numero total de candidatos por nivel e região

## Passo 1 — Probabilidade A Priori: P(Região)

Quantos % dos alunos são de cada região?

```
P(Região) = número de alunos da região / total de alunos
```


In [ ]:
total_candidatos = len(df)

# Conta alunos por região
contagem_regiao = df['Regiao'].value_counts().sort_index()

# Calcula a probabilidade a priori
priori = {}
for regiao in contagem_regiao.index:
    priori[regiao] = contagem_regiao[regiao] / total_candidatos

print("PROBABILIDADE A PRIORI  P(Região)")
for regiao, prob in priori.items():
    print(f"  {regiao:<13}: {prob:.4f}  ({prob*100:.2f}%)")

PROBABILIDADE A PRIORI  P(Região)
  Centro-Oeste : 0.0781  (7.81%)
  Nordeste     : 0.3627  (36.27%)
  Norte        : 0.1046  (10.46%)
  Sudeste      : 0.3407  (34.07%)
  Sul          : 0.1139  (11.39%)


## Passo 2 — Verossimilhança: P(Nível | Região)

Dentro de cada região, qual % dos alunos tem cada nível?

```
P(Nível | Região) = alunos com esse nível nessa região / total de alunos dessa região
```


In [ ]:
niveis = ['Muito Baixo', 'Baixo', 'Médio', 'Alto', 'Excelente']
regioes = sorted(df["Regiao"].unique())

tabela = pd.crosstab(df["Regiao"], df["Nivel"])

verossimilhanca = {}
for regiao in regioes:
    total_regiao = tabela.loc[regiao].sum()
    verossimilhanca[regiao] = {}
    for nivel in niveis:
        verossimilhanca[regiao][nivel] = tabela.loc[regiao, nivel] / total_regiao

print("Verossimilhança P(Nivel | Região)")
print(f"{'Região':<16}", end="")
for nivel in niveis:
    print(f"{nivel:>12}", end="")
print()

for regiao in regioes:
    print(f" {regiao:<13}", end="")
    for nivel in niveis:
        print(f"{verossimilhanca[regiao][nivel]:>12.4f}", end="")
    print()
    

Verossimilhança P(Nivel | Região)
Região           Muito Baixo       Baixo       Médio        Alto   Excelente
 Centro-Oeste       0.1021      0.4265      0.3355      0.1289      0.0071
 Nordeste           0.1456      0.4588      0.2952      0.0957      0.0047
 Norte              0.1847      0.5027      0.2431      0.0671      0.0024
 Sudeste            0.0671      0.3805      0.3865      0.1585      0.0074
 Sul                0.0719      0.4204      0.3722      0.1296      0.0059


Evidência

In [ ]:
contagem_nivel = df['Nivel'].value_counts()

evidencia = {}
for nivel in niveis:
    evidencia[nivel] = contagem_nivel[nivel] / total_candidatos

print("EVIDÊNCIA  P(Nível)")
print()
for nivel, prob in evidencia.items():
    print(f"  {nivel:<14}: {prob:.4f}  ({prob*100:.2f}%)")


EVIDÊNCIA  P(Nível)

  Muito Baixo   : 0.1112  (11.12%)
  Baixo         : 0.4298  (42.98%)
  Médio         : 0.3328  (33.28%)
  Alto          : 0.1205  (12.05%)
  Excelente     : 0.0057  (0.57%)


Teorema de Bayes

In [ ]:
posteriori = {}
for nivel in niveis:
    posteriori[nivel] = {}
    for regiao in regioes:
        numerador   = verossimilhanca[regiao][nivel] * priori[regiao]
        denominador = evidencia[nivel]
        posteriori[nivel][regiao] = numerador / denominador

print("PROBABILIDADE A POSTERIORI  P(Região | Nível)")
print(f"{'Nível':<16}", end="")
for regiao in regioes:
    print(f"{regiao:>14}", end="")
print()
print("-" * 75)
for nivel in niveis:
    print(f"  {nivel:<14}", end="")
    for regiao in regioes:
        print(f"{posteriori[nivel][regiao]:>14.4f}", end="")
    print()

PROBABILIDADE A POSTERIORI  P(Região | Nível)
Nível             Centro-Oeste      Nordeste         Norte       Sudeste           Sul
---------------------------------------------------------------------------
  Muito Baixo           0.0717        0.4751        0.1738        0.2057        0.0737
  Baixo                 0.0775        0.3871        0.1224        0.3016        0.1114
  Médio                 0.0787        0.3217        0.0764        0.3957        0.1274
  Alto                  0.0835        0.2879        0.0583        0.4478        0.1224
  Excelente             0.0964        0.2999        0.0436        0.4418        0.1183


In [ ]:
niveis_validos = ['Muito Baixo', 'Baixo', 'Médio', 'Alto', 'Excelente']

print("Niveis Validos")
for i, n in enumerate(niveis_validos, 1):
    print(f" {i}. {n} ")

escolha = input("\nDigite o numero do nível: ").strip()

if not escolha.isdigit() or int(escolha) not in range(1, len(niveis_validos) + 1):
     print(f"\nOpção inválida! Digite um número de 1 a {len(niveis_validos)}.")
else:
    nivel_escolhido = niveis_validos[int(escolha) - 1]
    resultado = posteriori[nivel_escolhido]    

    resultado_ordenado = sorted(resultado.items(), key=lambda x: x[1], reverse=True)
    regiao_predita = resultado_ordenado[0][0]

    print(f"Nível informado: {nivel_escolhido}")
    print()
    print(f"{'Região':<16} {'P(Região|Nível)':>18}")
    print()

    for regiao, prob in resultado_ordenado:
        marca = " <- PREDIÇÃO" if regiao == regiao_predita else ""
        print(f"  {regiao:<14} {prob:>14.4f}  ({prob*100:.2f}%){marca}")
    print()
    print(f"Resposta: aluno com nível '{nivel_escolhido}' → mais provável ser do {regiao_predita}")
    print(f"Confiança: {resultado[regiao_predita]*100:.2f}%")


Niveis Validos
 1. Muito Baixo 
 2. Baixo 
 3. Médio 
 4. Alto 
 5. Excelente 
Nível informado: Baixo

Região              P(Região|Nível)

  Nordeste               0.3871  (38.71%) <- PREDIÇÃO
  Sudeste                0.3016  (30.16%)
  Norte                  0.1224  (12.24%)
  Sul                    0.1114  (11.14%)
  Centro-Oeste           0.0775  (7.75%)

Resposta: aluno com nível 'Baixo' → mais provável ser do Nordeste
Confiança: 38.71%


In [ ]:
regioes_validas = sorted(df['Regiao'].unique())

print("Regiões disponíveis:")
for i, r in enumerate(regioes_validas, 1):
    print(f"  {i}. {r}")

escolha = input("\nDigite o número da região: ").strip()

if not escolha.isdigit() or int(escolha) not in range(1, len(regioes_validas) + 1):
    print(f"\nOpção inválida! Digite um número de 1 a {len(regioes_validas)}.")
else:
    regiao_escolhida = regioes_validas[int(escolha) - 1]

    resultado = verossimilhanca[regiao_escolhida]

    resultado_ordenado = sorted(resultado.items(), key=lambda x: x[1], reverse=True)
    nivel_predito = resultado_ordenado[0][0]

    print(f"\nRegião informada: {regiao_escolhida}")
    print(f"{'Nível':<16} {'P(Nível|Região)':>18}")
    for nivel, prob in resultado_ordenado:
        marca = " <- PREDIÇÃO" if nivel == nivel_predito else ""
        print(f"  {nivel:<14} {prob:>14.4f}  ({prob*100:.2f}%){marca}")
    print(f"Resposta: aluno da região '{regiao_escolhida}' → nível mais provável: {nivel_predito}")
    print(f"Confiança: {resultado[nivel_predito]*100:.2f}%")

Regiões disponíveis:
  1. Centro-Oeste
  2. Nordeste
  3. Norte
  4. Sudeste
  5. Sul

Região informada: Nordeste
Nível               P(Nível|Região)
  Baixo                  0.4588  (45.88%) <- PREDIÇÃO
  Médio                  0.2952  (29.52%)
  Muito Baixo            0.1456  (14.56%)
  Alto                   0.0957  (9.57%)
  Excelente              0.0047  (0.47%)
Resposta: aluno da região 'Nordeste' → nível mais provável: Baixo
Confiança: 45.88%


Usado IA para fazer metodo de acuracia do Teorema de bayes

In [ ]:
# CALCULANDO A "ACURÁCIA" DO TEOREMA DE BAYES

print("\n" + "="*70)
print("CALCULANDO A ACURÁCIA DO TEOREMA DE BAYES")
print("="*70)

# Usando a posteriori que você já calculou
# posteriori[nivel][regiao] = P(Região | Nível)

# Para cada nível, a região predita é aquela com maior P(Região|Nível)
regiao_predita_por_nivel = {}

for nivel in niveis:
    # Encontra a região com maior probabilidade a posteriori para este nível
    regiao_predita = max(posteriori[nivel].items(), key=lambda x: x[1])[0]
    regiao_predita_por_nivel[nivel] = regiao_predita
    print(f"\n  Nível '{nivel}':")
    print(f"    Região predita pelo Bayes: {regiao_predita}")
    print(f"    Confiança: {posteriori[nivel][regiao_predita]*100:.2f}%")

# Agora, vamos calcular a acurácia comparando com os dados reais
print("\n" + "-"*70)
print("COMPARANDO COM OS DADOS REAIS:")
print("-"*70)

# Contagem de acertos
acertos = 0
total = len(df)

# Para cada linha do dataset, verifica se a predição do Bayes acertou
for idx, row in df.iterrows():
    nivel_real = row['Nivel']
    regiao_real = row['Regiao']
    
    # Região predita pelo Bayes para este nível
    regiao_predita = regiao_predita_por_nivel[nivel_real]
    
    if regiao_predita == regiao_real:
        acertos += 1

acuracia_bayes = acertos / total

print(f"\n ACURÁCIA DO TEOREMA DE BAYES: {acuracia_bayes:.4f} ({acuracia_bayes*100:.2f}%)")
print(f"   Acertos: {acertos} de {total} registros")

print("\n" + "-"*70)
print("INTERPRETAÇÃO:")
print("-"*70)
print(f"  O Bayes acertaria {acertos} vezes se usássemos ele para classificar")
print(f"  todos os {total} alunos baseado APENAS no nível de desempenho.")


CALCULANDO A ACURÁCIA DO TEOREMA DE BAYES

  Nível 'Muito Baixo':
    Região predita pelo Bayes: Nordeste
    Confiança: 47.51%

  Nível 'Baixo':
    Região predita pelo Bayes: Nordeste
    Confiança: 38.71%

  Nível 'Médio':
    Região predita pelo Bayes: Sudeste
    Confiança: 39.57%

  Nível 'Alto':
    Região predita pelo Bayes: Sudeste
    Confiança: 44.78%

  Nível 'Excelente':
    Região predita pelo Bayes: Sudeste
    Confiança: 44.18%

----------------------------------------------------------------------
COMPARANDO COM OS DADOS REAIS:
----------------------------------------------------------------------

 ACURÁCIA DO TEOREMA DE BAYES: 0.4074 (40.74%)
   Acertos: 1142167 de 2803482 registros

----------------------------------------------------------------------
INTERPRETAÇÃO:
----------------------------------------------------------------------
  O Bayes acertaria 1142167 vezes se usássemos ele para classificar
  todos os 2803482 alunos baseado APENAS no nível de desempenh